# Stages of inference

How do attention and residual-stream statistics change through a recurrent model?
Separate attention-pattern measurements from residual measurements, following the
shared main-paper and appendix model groups. Compare the full forward pass and the
layer profile within each recurrence, including matched feedforward controls.
A final MLP ablation tests the first token's role in forming attention stages.

`mode="quick"` defaults to one seeded example and saves only in the run directory.
`mode="full"` defaults to 256 examples and also updates the paper figures. Both
modes use the same model–recurrence pairs and figures; SLURM selects full mode.
An explicit `limit` overrides the sample count, independently of export policy.

In [ ]:
mode = "quick"
dataset = "gsm8k"  # "gsm8k" or "hellaswag"
# Main/appendix models come from paper_models.py; all required pairs are resolved below.
profile_recurrences = 32
mixing_pair = ("retrofitted_llama", 8)  # Retained mixing-score illustration.
ouro_recurrences = 4  # Ouro keeps four recurrences in both short-run comparisons.
long_recurrences = 128  # Main models only; no residual entropy for these runs.
ablation_pair = ("retrofitted_llama", 32)
ablation_hook = "prelude_blocks.1.mlp.hook_out"
ablation_position = 0  # First token; block 1 is the second prelude block.
limit = None  # Default: 1 (quick), 256 (full); 0 means the entire split.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.
source_run = None  # Optional completed run directory with matching scientific settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from dataclasses import asdict, replace
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.lines import Line2D

from experiments.cache import ExperimentCache
from experiments.paper_models import PAPER_MODEL_GROUPS, paper_model_pairs
from experiments.saved_runs import SavedRun
from experiments.support import (
    BASELINE_FOR,
    COLORMAPS,
    MODEL_LABELS,
    add_depth_colorbar,
    add_prelude_coda_connectors,
    add_prelude_coda_legend,
    collect_metrics,
    configure_experiment,
    export_paper_figures,
    get_figsize,
    model_subplots,
    save_figure,
    save_results,
    summarize_stages,
)

plt.style.use(ROOT / "plots" / "matlab.mplstyle")


def short_run_pairs(recurrences):
    """Resolve shared paper groups, keeping Ouro's short rollout at four."""
    pairs = [
        (name, ouro_recurrences if name == "ouro_1_4b" else recurrences)
        for name in dict.fromkeys(PAPER_MODEL_GROUPS["all"])
    ]
    return paper_model_pairs(recurrences, pairs)


# Derive selections after Papermill overrides; repeated panels share one collection.
short_runs, figure_pairs = short_run_pairs(profile_recurrences)
long_pairs = [(name, long_recurrences) for name in PAPER_MODEL_GROUPS["main"]]
mixing_pair = tuple(mixing_pair)
ablation_pair = tuple(ablation_pair)
baseline_pairs = list(
    dict.fromkeys((BASELINE_FOR[name], 1) for name, _ in [*short_runs, ablation_pair])
)
model_recurrences = list(
    dict.fromkeys(
        [*short_runs, *long_pairs, *baseline_pairs, mixing_pair, ablation_pair]
    )
)
# Only within-recurrence residual profiles need spectral entropy.
entropy_pairs = [*short_runs, *baseline_pairs]
sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
configs = {
    (name, recurrences): configure_experiment(
        experiment="stages_of_inference",
        dataset=dataset,
        mode=mode,
        models=[name],
        num_steps=recurrences,
        compute_residual_entropy=(name, recurrences) in entropy_pairs,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
ATTENTION_METRICS = {
    "sink_rates": "Sink rate",
    "mixing_scores": "Mixing score",
    "colsum_concentrations": "Column concentration",
}
RESIDUAL_METRICS = {
    "residual_entropy": "Residual entropy",
    "residual_norms": "First-token residual norm",
    "residual_norms_last": "Last-token residual norm",
    "residual_norms_mean": "Mean residual norm",
}
METRICS = ATTENTION_METRICS | RESIDUAL_METRICS


def zero_first_token_mlp(value, hook):
    """Remove the selected prelude MLP contribution at the first token only."""
    value[:, ablation_position, :] = 0.0
    return value


def aggregate_stages(examples):
    """Give each example equal weight; bands show one sample standard deviation."""
    topology = {
        key: examples[0][key]
        for key in ("layers", "virtual_layers", "recurrent_layers")
    }
    curves = {}
    for field in examples[0]["curves"]:
        values = np.asarray([example["curves"][field] for example in examples])
        curves[field] = {
            "mean": values.mean(axis=0),
            "std": values.std(axis=0, ddof=1 if len(examples) > 1 else 0),
        }
    return {**topology, "curves": curves}


def recurrence_groups(data):
    """Use recorded layer identities to separate prelude, recurrences, and coda."""
    layers = np.asarray(data["layers"])
    core = data["recurrent_layers"]
    if not core:
        return {"Feedforward": np.arange(len(layers))}
    groups = {}
    if np.any(layers < min(core)):
        groups["Prelude"] = np.flatnonzero(layers < min(core))
    by_layer = [np.flatnonzero(layers == layer) for layer in core]
    for recurrence, indices in enumerate(np.asarray(by_layer).T):
        groups[f"R{recurrence}"] = indices
    if np.any(layers > max(core)):
        groups["Coda"] = np.flatnonzero(layers > max(core))
    return groups

## Generate and reduce

The shared model groups determine the minimum required runs. All short-run
comparisons use 32 recurrences, except Ouro, which retains four. Retrofitted Llama
also runs at eight for the retained mixing-score illustration.
Only main-paper models also run at 128 recurrences. Parcae 770M is in the main
group; Parcae 140M and 370M are in the appendix.

Llama 3.2 1B accompanies Ouro, Huginn, and Retrofitted Llama. OLMo2 and TinyLlama
accompany their respective retrofits. Each Parcae model uses its matched, trained
one-pass Parcae control. Collect each baseline once and reuse it across figures.
All runs use the same seeded examples with each checkpoint's native tokenization.

Retrofitted Llama at 32 recurrences additionally runs with the selected prelude
MLP output zeroed at position 0. The intervention is active only during that run.
Its attention-only reduction disables residual entropy, as do the long rollouts
and the eight-recurrence illustration. Pairwise and convergence comparisons are also disabled.
Retain per-example scalar curves and metadata in `overview.pt`, then release models
and collected metrics. Every later figure cell reuses these summaries.

In [ ]:
settings = asdict(config)
del settings["models"], settings["num_steps"], settings["compute_residual_entropy"]
settings.update(model_recurrences=model_recurrences, output_dir=str(config.output_dir))
settings["entropy_pairs"] = entropy_pairs
settings["figure_pairs"] = figure_pairs
settings["long_pairs"] = long_pairs
settings["mixing_pair"] = mixing_pair
settings["ablation"] = {
    "model": ablation_pair[0],
    "recurrences": ablation_pair[1],
    "hook": ablation_hook,
    "position": ablation_position,
    "value": 0.0,
}
settings["versions"] = {
    name: version(name) for name in ("torch", "transformer-lens", "transformers")
}
settings["cache"] = result_cache.settings
previous = SavedRun(source_run, settings, analysis_version=analysis_version)
previous.check_pairs(settings)
settings.update(analysis_version=analysis_version, source_run=source_run)
run_dir = save_results({}, settings, config.output_dir)
for folder in ("similar_behaviour", "recurrent"):
    (run_dir / folder).mkdir()

summaries, run_info = {}, {}
for pair, pair_config in configs.items():
    if pair in previous:
        summaries[pair] = previous.data["summaries"][pair]
        run_info[pair] = previous.info(pair)
        continue
    name, recurrences = pair
    results, info = collect_metrics(
        pair_config,
        summarize_stages,
        cache=result_cache,
        analysis_version=analysis_version,
    )
    summaries[pair] = results[name]
    run_info[pair] = info
    del results

if "ablation" in previous.data:
    ablation_summaries = previous.data["ablation"]["summaries"]
    ablation_info = {
        **previous.data["ablation"]["run"],
        "source_run": str(previous.directory),
    }
else:
    results, ablation_info = collect_metrics(
        replace(configs[ablation_pair], compute_residual_entropy=False),
        summarize_stages,
        fwd_hooks=[(ablation_hook, zero_first_token_mlp)],
        cache=result_cache,
        analysis_version=analysis_version,
        intervention=settings["ablation"],
    )
    ablation_summaries = results[ablation_pair[0]]
    ablation_info["ablation"] = settings["ablation"]
    del results

stages = {pair: aggregate_stages(examples) for pair, examples in summaries.items()}
ablation_stages = aggregate_stages(ablation_summaries)
torch.save(
    {
        "summaries": summaries,
        "run": run_info,
        "figure_pairs": settings["figure_pairs"],
        "ablation": {"summaries": ablation_summaries, "run": ablation_info},
    },
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## What the curves measure

| Measurement | Interpretation |
| --- | --- |
| Sink rate | Fraction of heads whose mean attention to the first token is at least 0.3. |
| Mixing score | Attention-row entropy divided by its causal maximum, averaged over query rows excluding the first, then over heads. |
| Column concentration | One minus the normalised entropy of attention received across key positions, averaged over heads. |
| Residual entropy | $-\sum_i p_i\log p_i$, where $p_i$ is the normalised squared singular value of the token-by-feature residual matrix. The complete spectrum is used. |
| First / last-token residual norm | L2 norm at the first / last position of the evaluated input. |
| Mean residual norm | Compute an L2 norm at **each token**, then average those norms within the example; this is not the norm of the mean residual vector. |

Average heads **within each example first**, then give each example equal weight.
Mean residual norms likewise average tokens before examples, so longer sequences
do not receive extra weight. Inputs include the likelihood task's context and
continuation: the first token can be BOS, and the last token belongs to the
continuation. Native tokenization means positions can differ between models.
Shading shows one sample standard deviation across examples, or zero for one
example. Residual entropy describes the residual matrix, not vocabulary entropy.

A **virtual layer** is one layer application; a **layer** identifies weights reused
across recurrences. Percentage depth divides the recorded index by its maximum,
placing the first and last layers at 0% and 100%. Layer depth includes prelude and
coda; recurrence numbers and layer indices are zero-based. Identify these regions
from recorded layer metadata, rather than fixed offsets.

Within-recurrence profiles show how processing changes through the block. Each
feedforward control uses its own normalised layer-depth axis. Neither a visible
stage nor a stable statistic alone establishes a fixed point of the residual
stream or better task performance.

In [ ]:
def depth_positions(data, *, by_layer=False):
    positions = np.asarray(data["layers"] if by_layer else data["virtual_layers"])
    return positions / max(positions.max(), 1) * 100


def draw_curve(ax, data, field, *, indices=None, x=None, band=True, **style):
    indices = np.arange(len(data["layers"])) if indices is None else indices
    x = np.asarray(data["virtual_layers"])[indices] if x is None else x
    mean = data["curves"][field]["mean"][indices]
    std = data["curves"][field]["std"][indices]
    (line,) = ax.plot(x, mean, **style)
    if band:
        ax.fill_between(x, mean - std, mean + std, color=line.get_color(), alpha=0.2)
    return line


def draw_layers(ax, pair, field, *, connect=False):
    """Connect layer trajectories in the retained mixing-score illustration."""
    data = stages[pair]
    if connect:
        draw_curve(
            ax,
            data,
            field,
            band=False,
            color="black",
            linestyle="--",
            label="True layer order",
        )
    for layer in data["recurrent_layers"]:
        indices = np.flatnonzero(np.asarray(data["layers"]) == layer)
        draw_curve(
            ax,
            data,
            field,
            indices=indices,
            marker="x",
            label=f"Layer {layer}",
            linewidth=2,
        )
    ax.set_xlabel("Virtual layer")


def draw_profiles(ax, pair, field, *, band=True, data=None):
    """Compare each recurrence's layer profile with its feedforward baseline."""
    data = stages[pair] if data is None else data
    groups = recurrence_groups(data)
    n_recurrences = sum(key.startswith("R") for key in groups)
    x = depth_positions(data, by_layer=True)
    lines = {}
    for key, indices in groups.items():
        if key in {"Prelude", "Coda"}:
            color, label = ("orange" if key == "Prelude" else "black"), key
        else:
            color = plt.colormaps[COLORMAPS["recurrences"]](
                int(key[1:]) / max(n_recurrences - 1, 1)
            )
            label = None
        lines[key] = draw_curve(
            ax,
            data,
            field,
            indices=indices,
            x=x[indices],
            band=band,
            color=color,
            label=label,
            linewidth=2,
        )
    name = BASELINE_FOR[pair[0]]
    baseline = stages[(name, 1)]
    draw_curve(
        ax,
        baseline,
        field,
        x=depth_positions(baseline, by_layer=True),
        color="red",
        linestyle="--",
        label=MODEL_LABELS[name],
        linewidth=2,
    )
    add_prelude_coda_connectors(ax, lines)
    ax.set_xlabel("Layer depth (%)")

In [ ]:
MODEL_COLORS = {
    name: plt.colormaps["tab10"](index)
    for index, name in enumerate(dict.fromkeys(PAPER_MODEL_GROUPS["all"]))
}


def plot_attention_depth(group):
    fig, axes = plt.subplots(
        1, 3, sharex=True, layout="constrained", figsize=get_figsize(True, (1, 3), 2)
    )
    for ax, (field, label) in zip(axes, ATTENTION_METRICS.items(), strict=True):
        for pair in figure_pairs[group]:
            draw_curve(
                ax,
                stages[pair],
                field,
                x=depth_positions(stages[pair]),
                color=MODEL_COLORS[pair[0]],
                label=MODEL_LABELS[pair[0]],
            )
        ax.set(xlabel="Virtual layer depth (%)", ylabel=label)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncols=4)
    save_figure(
        fig, run_dir, f"attention_metrics_in_total_depth_{group}", bbox_inches=None
    )


def plot_attention_profiles(pair, filename, *, band=True, data=None):
    fig, axes = plt.subplots(
        1, 3, sharex=True, layout="constrained", figsize=get_figsize(True, (1, 3), 1.7)
    )
    for ax, (field, label) in zip(axes, ATTENTION_METRICS.items(), strict=True):
        draw_profiles(ax, pair, field, band=band, data=data)
        ax.set_ylabel(label)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncols=len(labels))
    add_depth_colorbar(fig, axes, "recurrence")
    save_figure(fig, run_dir, filename, bbox_inches=None)


def plot_profile_grid(pairs, field, filename, *, shared_legend=False):
    fig, axes = model_subplots(len(pairs), height_per_subplot=1.8)
    for index, (ax, pair) in enumerate(zip(axes, pairs, strict=True)):
        draw_profiles(ax, pair, field)
        ax.set_title(MODEL_LABELS[pair[0]])
        if index % 4 == 0:
            ax.set_ylabel(METRICS[field])
        ax.locator_params(axis="x", nbins=3)
        if not shared_legend:
            ax.legend()
    if shared_legend:
        add_prelude_coda_legend(fig)
    add_depth_colorbar(fig, axes, "recurrence")
    save_figure(fig, run_dir, filename, bbox_inches=None)

In [ ]:
def plot_residual_norms(group):
    fig, axes = plt.subplots(
        1, 2, sharex=True, layout="constrained", figsize=get_figsize(True, (1, 2), 2)
    )
    for pair in figure_pairs[group]:
        data = stages[pair]
        style = dict(x=depth_positions(data), color=MODEL_COLORS[pair[0]], linewidth=2)
        draw_curve(axes[0], data, "residual_norms", linestyle="-", **style)
        draw_curve(axes[0], data, "residual_norms_last", linestyle="--", **style)
        draw_curve(
            axes[1], data, "residual_norms_mean", label=MODEL_LABELS[pair[0]], **style
        )
    for ax, title in zip(
        axes, ("First and last tokens", "Mean over all tokens"), strict=True
    ):
        ax.set(
            title=title,
            xlabel="Virtual layer depth (%)",
            ylabel="Residual norm",
            yscale="log",
        )
    axes[0].legend(
        handles=[
            Line2D([], [], color="black", linestyle="-", label="First token"),
            Line2D([], [], color="black", linestyle="--", label="Last token"),
        ]
    )
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncols=4)
    save_figure(fig, run_dir, f"residual_norm_comparison_{group}", bbox_inches=None)

## Attention metrics in total depth

Main-paper and appendix models are shown in their shared order, without
feedforward controls. Ouro uses four recurrences; the other models use 32.
Each model keeps the same colour in the residual-norm comparisons below.

In [ ]:
plot_attention_depth("main")

In [ ]:
plot_attention_depth("appendix")

### Mixing-score trajectories

The single-column Retrofitted Llama illustration retains eight recurrences.
Coloured lines connect the same layer across recurrences; the dashed black line
follows the true execution order.

In [ ]:
fig, ax = plt.subplots(
    figsize=get_figsize(False, (1, 1), height_per_subplot=1.9),
    layout="constrained",
)
draw_layers(ax, mixing_pair, "mixing_scores", connect=True)
ax.set_ylabel(METRICS["mixing_scores"])
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=4)
save_figure(
    fig, run_dir, "similar_behaviour/similar_behaviour_summarised", bbox_inches=None
)

## Residual norms in total depth

Colours identify models; solid and dashed lines identify the first and last
tokens. The right panel averages norms across all tokens within each example.
Both panels retain the logarithmic scale and show example-standard-deviation bands.

In [ ]:
plot_residual_norms("main")

In [ ]:
plot_residual_norms("appendix")

## Attention Metrics Within Each Recurrence

Each model has a full-width figure containing sink rate, mixing score, and column
concentration. All models, including Ouro, use the same early-to-late recurrence
colourbar. Orange and black mark prelude and coda; the dashed red line is the
matched feedforward control. Ouro uses four recurrences and the other models use
32. The main-paper mixing-score summary reuses these same four/32-step runs.

In [ ]:
for pair in short_runs:
    plot_attention_profiles(pair, f"recurrent/{MODEL_LABELS[pair[0]]}")

In [ ]:
plot_profile_grid(
    figure_pairs["main"], "mixing_scores", "recurrent/summarised", shared_legend=True
)

## Residual Metrics Within Each Recurrence

Two full-width `all` figures separate residual entropy from mean residual norm.
Each has the main-paper models in the first row and appendix models in the second,
with Parcae 770M in the main row. Both use the same recurrence profiles and
matched controls as the attention figures: four recurrences for
Ouro and 32 for the others. Axes have independent vertical scales for each model.

In [ ]:
plot_profile_grid(
    figure_pairs["all"], "residual_entropy", "recurrent/residual_entropy_all"
)

In [ ]:
plot_profile_grid(
    figure_pairs["all"], "residual_norms_mean", "recurrent/residual_norm_mean_all"
)

## Stability over longer rollouts

The retained single-column summary compares within-recurrence mixing profiles
for Retrofitted Llama and Ouro at 128 recurrences, with uncertainty bands.
Full-width, attention-only figures then cover every main-paper model in order.
Recurrent curves show means only to keep 128 overlaid profiles readable; the
feedforward control retains its uncertainty band. The final observed recurrence
is a finite-rollout reference, not a demonstrated fixed point.

In [ ]:
pairs = [("retrofitted_llama", long_recurrences), ("ouro_1_4b", long_recurrences)]
fig, axes = plt.subplots(
    1,
    2,
    sharex=True,
    layout="constrained",
    figsize=get_figsize(False, (1, 2), height_per_subplot=1.5),
)
for ax, pair in zip(axes, pairs, strict=True):
    draw_profiles(ax, pair, "mixing_scores")
    ax.set_title(MODEL_LABELS[pair[0]].replace("Retrofitted ", "R."))
axes[0].set_ylabel(METRICS["mixing_scores"])
add_depth_colorbar(fig, axes, "recurrence")
save_figure(
    fig,
    run_dir,
    "similar_behaviour/stability_in_percentage_depth_summarised",
    bbox_inches=None,
)

In [ ]:
for pair in long_pairs:
    plot_attention_profiles(
        pair,
        f"recurrent/{MODEL_LABELS[pair[0]]} ({pair[1]} steps)_to_fixed_point",
        band=False,
    )

## First-token MLP ablation

Zero the MLP output at **prelude block 1** (the second prelude block), at **token
position 0** only, on each forward pass of Retrofitted Llama with 32 recurrences.
This removes the MLP contribution before the recurrent core runs. Native
tokenization is retained; position 0 is BOS when the tokenizer prepends it.

Compare the three attention profiles with the unablated Retrofitted Llama figure
above to test their dependence on this early first-token contribution. Both
conditions use the same seeded examples, native tokenization, recurrence colours,
matched Llama 3.2 1B control, and example-standard-deviation bands.

In [ ]:
plot_attention_profiles(
    ablation_pair, "ablation_retrofitted_llama_bos_mlp", data=ablation_stages
)

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir, dataset=config.dataset)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")